# Temporal Capability Tracking (2023-2026)

This notebook tracks intelligence progression and price compression across model generations using `release_year`.

Approach:
- Use both **median** and **mean** yearly statistics
- Treat **median** as primary (robust to outliers)
- Use **mean** as secondary context (tail effects)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

csv_path = "../data/llm_price_performance_tracker_2026-03-31.csv"
df = pd.read_csv(csv_path)

# Standardize column names for consistent access
rename_map = {
    c: c.strip().lower().replace("\n", "_").replace(" ", "_").replace("-", "_")
    for c in df.columns
}
df = df.rename(columns=rename_map)

for col in ["release_year", "aa_intelligence_index", "blended_cost_usd_per_1m"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df.head(3)

## Step 1: Build temporal analysis subset
Filter to years 2023-2026 and keep rows needed for intelligence/cost trend analysis.

In [ ]:
years = [2023, 2024, 2025, 2026]

temporal_df = df[
    (df["release_year"].isin(years))
    & (df["aa_intelligence_index"].notna())
    & (df["blended_cost_usd_per_1m"].notna())
].copy()

temporal_df["release_year"] = temporal_df["release_year"].astype(int)

print(f"Rows in temporal subset: {len(temporal_df)}")
print("\nRows by year:")
display(temporal_df["release_year"].value_counts().sort_index())

## Step 2: Intelligence progression by year (median + mean)
Median is the primary trend; mean is included as secondary context.

In [ ]:
yearly_intel = (
    temporal_df.groupby("release_year")["aa_intelligence_index"]
    .agg(median_intelligence="median", mean_intelligence="mean", model_count="count")
    .reset_index()
)

plt.figure(figsize=(9, 5))
plt.plot(
    yearly_intel["release_year"],
    yearly_intel["median_intelligence"],
    marker="o",
    linewidth=2.5,
    label="Median intelligence",
)
plt.plot(
    yearly_intel["release_year"],
    yearly_intel["mean_intelligence"],
    marker="o",
    linestyle="--",
    linewidth=2,
    label="Mean intelligence",
)
plt.title("Intelligence Progression by Release Year")
plt.xlabel("Release Year")
plt.ylabel("AA Intelligence Index")
plt.xticks(years)
plt.legend()
plt.tight_layout()
plt.show()

# Distribution view by year
plt.figure(figsize=(9, 5))
sns.boxplot(data=temporal_df, x="release_year", y="aa_intelligence_index", color="#8ecae6")
sns.stripplot(data=temporal_df, x="release_year", y="aa_intelligence_index", color="#023047", alpha=0.35, size=3)
plt.title("Intelligence Distribution by Release Year")
plt.xlabel("Release Year")
plt.ylabel("AA Intelligence Index")
plt.tight_layout()
plt.show()

yearly_intel

## Step 3: Price compression by year (median + mean)
Costs are plotted on a log scale because pricing is long-tailed.

In [ ]:
yearly_cost = (
    temporal_df.groupby("release_year")["blended_cost_usd_per_1m"]
    .agg(median_cost="median", mean_cost="mean", model_count="count")
    .reset_index()
)

yearly_cost["yoy_median_cost_pct"] = yearly_cost["median_cost"].pct_change() * 100

plt.figure(figsize=(9, 5))
plt.plot(
    yearly_cost["release_year"],
    yearly_cost["median_cost"],
    marker="o",
    linewidth=2.5,
    label="Median blended cost",
)
plt.plot(
    yearly_cost["release_year"],
    yearly_cost["mean_cost"],
    marker="o",
    linestyle="--",
    linewidth=2,
    label="Mean blended cost",
)
plt.yscale("log")
plt.title("Price Compression by Release Year (log scale)")
plt.xlabel("Release Year")
plt.ylabel("Blended Cost (USD per 1M tokens, log scale)")
plt.xticks(years)
plt.legend()
plt.tight_layout()
plt.show()

yearly_cost

## Step 4: Capability-price frontier over time
Scatter plot of cost vs intelligence colored by year to visualize movement toward better quality at lower cost.

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.scatterplot(
    data=temporal_df,
    x="blended_cost_usd_per_1m",
    y="aa_intelligence_index",
    hue="release_year",
    palette="viridis",
    alpha=0.75,
)
ax.set_xscale("log")
ax.set_title("Capability-Price Frontier by Release Year")
ax.set_xlabel("Blended Cost (USD per 1M tokens, log scale)")
ax.set_ylabel("AA Intelligence Index")
plt.tight_layout()
plt.show()

## Step 5: Yearly summary and interpretation
A compact summary table and short narrative points.

In [ ]:
summary = (
    temporal_df.groupby("release_year")
    .agg(
        models=("model_name", "count"),
        median_intelligence=("aa_intelligence_index", "median"),
        mean_intelligence=("aa_intelligence_index", "mean"),
        median_cost=("blended_cost_usd_per_1m", "median"),
        mean_cost=("blended_cost_usd_per_1m", "mean"),
    )
    .reset_index()
)

summary["yoy_median_intelligence"] = summary["median_intelligence"].diff()
summary["yoy_median_cost_pct"] = summary["median_cost"].pct_change() * 100

display(summary.round(3))

print("Interpretation guide:")
print("- Median lines reflect the typical model each year.")
print("- Mean lines reveal tail effects from very expensive or very capable outliers.")
print("- Negative YoY median cost % indicates price compression.")
print("- Positive YoY median intelligence indicates capability progression.")